# Language-modeling lab: loss, perplexity and a bigram model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankitrgupta1/silicon-to-agents/blob/main/labs/lm-objective.ipynb)

Build the simplest possible language model, measure it the way real LLMs are measured, and sample from it. Companion to [The language-modeling objective](../03-foundations/lm-objective.md). Runs on CPU in seconds.

In [ ]:
try:
    import s2a
except ImportError:
    %pip install -q git+https://github.com/ankitrgupta1/silicon-to-agents

import math
import numpy as np
from s2a.lm import BigramModel, cross_entropy, perplexity, nats_to_bits

## 1. Data

We work with characters as tokens to keep things visible. Paste in your own text for better results: more text gives a better model.

In [ ]:
text = """
A language model reads text one token at a time and tries to guess what comes next.
When it guesses well, the loss is small. When it is surprised, the loss is large.
Training nudges the model so that the next token gets a little more probability each time.
After enough text, the model has learned spelling, grammar, facts and style, all from
the single goal of predicting the next token. That simple goal is what this lab measures.
The same idea scales from this tiny model to the largest models in the world: they are
trained to make the next token less surprising, and they are judged by how surprised they are.
""".strip()

chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
ids = [stoi[c] for c in text]
split = int(0.85 * len(ids))
train, val = ids[:split], ids[split:]
V = len(chars)
print(f"{len(ids)} characters, vocabulary of {V}, {len(train)} for training, {len(val)} held out")

## 2. Baselines: guessing uniformly, and guessing by frequency

A model that knows nothing spreads probability evenly, so its loss is `ln(V)` and its perplexity is exactly `V`.

In [ ]:
uniform = cross_entropy(np.zeros((len(val) - 1, V)), np.array(val[1:]))

freq = np.bincount(train, minlength=V) + 1.0          # unigram: how common each character is
unigram = cross_entropy(np.tile(np.log(freq / freq.sum()), (len(val) - 1, 1)), np.array(val[1:]))

for name, loss in [("uniform", uniform), ("unigram", unigram)]:
    print(f"{name:>8}: loss {loss:.3f} nats = {nats_to_bits(loss):.2f} bits, perplexity {perplexity(loss):.1f}")

## 3. A bigram model: use the previous character

Count how often each character follows each other one. Compare loss on the training text and on held-out text.

In [ ]:
for smoothing in [0.01, 0.1, 1.0]:
    m = BigramModel(V, smoothing=smoothing).fit(train)
    print(f"smoothing {smoothing:>4}: train loss {m.loss(train):.3f}, held-out loss {m.loss(val):.3f}, "
          f"held-out perplexity {perplexity(m.loss(val)):.1f}")

Held-out loss is what matters. Too little smoothing and the model is overconfident about pairs it saw once (overfitting); too much and it ignores what it learned.

## 4. Sampling, and what temperature does

In [ ]:
m = BigramModel(V, smoothing=0.1).fit(ids)
rng = np.random.default_rng(0)
for t in [0.3, 1.0, 2.0]:
    out = m.sample(stoi["T"], 120, rng=rng, temperature=t)
    print(f"temperature {t}:\n  " + "".join(chars[i] for i in out).replace("\n", " ") + "\n")

Low temperature repeats the most likely pairs; high temperature drifts toward noise. A bigram model only sees one character back, which is why the output looks word-like but meaningless. Transformers use attention to look back over thousands of tokens, and get much lower loss on the same objective.

## Try it

- Paste in a few pages of text. How low does held-out loss get?
- Build a trigram model (count pairs of previous characters). Does held-out loss improve, or does it overfit on this small text?
- Compute bits per character for each model. Shannon estimated English at roughly 1 bit per character. How far off is a bigram model?